# PTB-XL ECG Dataset - Data Exploration

**Course:** AAI-501 - Introduction to AI and Machine Learning  
**Project:** ECG Arrhythmia Classification  
**Part:** 1 - Data Preparation & EDA  
**Author:** Ashok Bhairwal

## Objectives
1. Load and understand the PTB-XL dataset structure
2. Explore metadata and patient demographics
3. Analyze diagnostic label distribution
4. Visualize sample ECG signals
5. Identify data quality issues

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
import ast
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Libraries loaded successfully!")

## 1. Load Dataset

In [ ]:
# Define paths
DATA_PATH = Path('../data/raw/ptb-xl')
SAMPLING_RATE = 100  # Use 100 Hz for faster processing

# Load metadata
df = pd.read_csv(DATA_PATH / 'ptbxl_database.csv', index_col='ecg_id')
print(f"Dataset shape: {df.shape}")
print(f"Total ECG records: {len(df)}")
print(f"Total patients: {df['patient_id'].nunique()}")

df.head()

In [ ]:
# Dataset info
print("\nDataset Information:")
print("=" * 50)
df.info()

In [ ]:
# Column names and descriptions
print("\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 2. Patient Demographics

In [ ]:
# Age distribution
print("Age Statistics:")
print(df['age'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age histogram
axes[0].hist(df['age'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Age Distribution')
axes[0].axvline(df['age'].median(), color='red', linestyle='--', label=f'Median: {df["age"].median():.1f}')
axes[0].legend()

# Age boxplot
axes[1].boxplot(df['age'].dropna(), vert=True)
axes[1].set_ylabel('Age (years)')
axes[1].set_title('Age Distribution (Boxplot)')

plt.tight_layout()
plt.show()

In [ ]:
# Sex distribution
print("\nSex Distribution:")
sex_counts = df['sex'].value_counts()
print(sex_counts)
print(f"\nPercentage:")
print(df['sex'].value_counts(normalize=True) * 100)

# Visualization
fig, ax = plt.subplots(figsize=(8, 6))
sex_counts.plot(kind='bar', ax=ax, color=['skyblue', 'lightcoral'])
ax.set_xlabel('Sex (0=Male, 1=Female)')
ax.set_ylabel('Count')
ax.set_title('Sex Distribution')
ax.set_xticklabels(['Male', 'Female'], rotation=0)
for i, v in enumerate(sex_counts):
    ax.text(i, v + 200, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Height and Weight
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['height'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Height (cm)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Height Distribution')

axes[1].hist(df['weight'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Weight (kg)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Weight Distribution')

plt.tight_layout()
plt.show()

print(f"Height - Mean: {df['height'].mean():.1f} cm, Median: {df['height'].median():.1f} cm")
print(f"Weight - Mean: {df['weight'].mean():.1f} kg, Median: {df['weight'].median():.1f} kg")

## 3. Diagnostic Labels Analysis

In [ ]:
# Load SCP statements
scp_statements = pd.read_csv(DATA_PATH / 'scp_statements.csv', index_col=0)
print(f"Total diagnostic statements: {len(scp_statements)}")
print(f"\nSCP Statements (first 10):")
scp_statements.head(10)

In [ ]:
# Parse SCP codes
df['scp_codes_dict'] = df['scp_codes'].apply(lambda x: ast.literal_eval(x))

# Map to diagnostic classes
def aggregate_diagnostic(scp_dict):
    agg_dict = {}
    for key in scp_dict.keys():
        if key in scp_statements.index:
            diag_class = scp_statements.loc[key, 'diagnostic_class']
            if pd.notna(diag_class):
                agg_dict[diag_class] = 1
    return agg_dict

df['diagnostic_superclass'] = df['scp_codes_dict'].apply(aggregate_diagnostic)

# Extract superclass labels
superclass_labels = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
for label in superclass_labels:
    df[label] = df['diagnostic_superclass'].apply(lambda x: 1 if label in x else 0)

print("Superclass Labels Distribution:")
for label in superclass_labels:
    print(f"{label}: {df[label].sum()}")

In [ ]:
# Visualize superclass distribution
superclass_counts = df[superclass_labels].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
superclass_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Diagnostic Superclass')
ax.set_ylabel('Count')
ax.set_title('Distribution of Diagnostic Superclasses')
ax.set_xticklabels(['NORM\n(Normal)', 'MI\n(Myocardial\nInfarction)', 
                    'STTC\n(ST/T Change)', 'CD\n(Conduction\nDisturbance)', 
                    'HYP\n(Hypertrophy)'], rotation=0)
for i, v in enumerate(superclass_counts):
    ax.text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-label analysis
df['num_labels'] = df[superclass_labels].sum(axis=1)
print("\nMulti-label Distribution:")
print(df['num_labels'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(8, 5))
df['num_labels'].value_counts().sort_index().plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_xlabel('Number of Labels per Record')
ax.set_ylabel('Count')
ax.set_title('Multi-label Distribution')
plt.tight_layout()
plt.show()

## 4. Signal Quality Assessment

In [ ]:
# Quality indicators
quality_cols = ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats']

print("Signal Quality Issues:")
for col in quality_cols:
    count = df[col].sum()
    pct = (count / len(df)) * 100
    print(f"{col:25s}: {count:5d} ({pct:5.2f}%)")

In [ ]:
# Visualize quality issues
quality_counts = df[quality_cols].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
quality_counts.plot(kind='barh', ax=ax, color='lightgreen', edgecolor='black')
ax.set_xlabel('Count')
ax.set_ylabel('Quality Issue')
ax.set_title('Signal Quality Issues Distribution')
for i, v in enumerate(quality_counts):
    ax.text(v + 50, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Load and Visualize Sample ECG Signals

In [ ]:
# Function to load ECG signal
def load_ecg_signal(ecg_id, sampling_rate=100):
    if sampling_rate == 100:
        path = DATA_PATH / df.loc[ecg_id, 'filename_lr']
    else:
        path = DATA_PATH / df.loc[ecg_id, 'filename_hr']
    signal, meta = wfdb.rdsamp(str(path))
    return signal, meta

# Test loading
sample_id = df.index[0]
signal, meta = load_ecg_signal(sample_id, SAMPLING_RATE)
print(f"Signal shape: {signal.shape}")
print(f"Sampling rate: {meta['fs']} Hz")
print(f"Signal names: {meta['sig_name']}")
print(f"Duration: {signal.shape[0] / meta['fs']:.1f} seconds")

In [ ]:
# Visualize sample ECG (12-lead)
lead_names = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
time = np.arange(signal.shape[0]) / SAMPLING_RATE

fig, axes = plt.subplots(12, 1, figsize=(14, 16), sharex=True)
fig.suptitle(f'12-Lead ECG - Sample ID: {sample_id}', fontsize=16, fontweight='bold')

for i, (ax, lead) in enumerate(zip(axes, lead_names)):
    ax.plot(time, signal[:, i], linewidth=0.8, color='black')
    ax.set_ylabel(lead, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([signal[:, i].min() - 0.2, signal[:, i].max() + 0.2])

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

In [ ]:
# Compare different diagnostic classes
fig, axes = plt.subplots(len(superclass_labels), 1, figsize=(14, 12))
fig.suptitle('Sample ECG Signals by Diagnostic Class (Lead II)', fontsize=16, fontweight='bold')

for i, label in enumerate(superclass_labels):
    # Get first sample with this label
    sample_ids = df[df[label] == 1].index
    if len(sample_ids) > 0:
        sample_id = sample_ids[0]
        signal, meta = load_ecg_signal(sample_id, SAMPLING_RATE)
        time = np.arange(signal.shape[0]) / SAMPLING_RATE
        
        axes[i].plot(time, signal[:, 1], linewidth=1, color='darkblue')  # Lead II
        axes[i].set_ylabel(f'{label}\n(mV)', fontweight='bold')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim([0, 10])

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

## 6. Missing Data Analysis

In [ ]:
# Missing data summary
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print("Missing Data Summary (Top 10):")
print(missing_df.head(10))

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
missing_df.head(15)['Percentage'].plot(kind='barh', ax=ax, color='salmon', edgecolor='black')
ax.set_xlabel('Missing Percentage (%)')
ax.set_ylabel('Column')
ax.set_title('Missing Data Analysis (Top 15 Columns)')
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
# Dataset summary
print("="*60)
print("PTB-XL DATASET SUMMARY")
print("="*60)
print(f"Total ECG Records: {len(df):,}")
print(f"Total Patients: {df['patient_id'].nunique():,}")
print(f"\nPatient Demographics:")
print(f"  Age Range: {df['age'].min():.0f} - {df['age'].max():.0f} years")
print(f"  Age Median: {df['age'].median():.0f} years")
print(f"  Male: {(df['sex']==0).sum():,} ({(df['sex']==0).sum()/len(df)*100:.1f}%)")
print(f"  Female: {(df['sex']==1).sum():,} ({(df['sex']==1).sum()/len(df)*100:.1f}%)")
print(f"\nDiagnostic Classes:")
for label in superclass_labels:
    count = df[label].sum()
    pct = (count / len(df)) * 100
    print(f"  {label:4s}: {count:5d} ({pct:5.2f}%)")
print(f"\nSignal Specifications:")
print(f"  Leads: 12 (I, II, III, AVR, AVL, AVF, V1-V6)")
print(f"  Sampling Rates: 100 Hz / 500 Hz")
print(f"  Duration: 10 seconds")
print(f"  Resolution: 16-bit")
print("="*60)

## 8. Save Processed Metadata

In [ ]:
# Save enhanced metadata
output_path = Path('../data/preprocessed')
output_path.mkdir(parents=True, exist_ok=True)

# Select relevant columns
cols_to_save = ['patient_id', 'age', 'sex', 'height', 'weight', 'filename_lr', 'filename_hr',
                'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats',
                'NORM', 'MI', 'STTC', 'CD', 'HYP', 'num_labels', 'strat_fold']

df_clean = df[cols_to_save].copy()
df_clean.to_csv(output_path / 'metadata_with_labels.csv')
print(f"Saved processed metadata to: {output_path / 'metadata_with_labels.csv'}")
print(f"Shape: {df_clean.shape}")

## Key Findings

### Dataset Characteristics:
- 21,799 ECG records from 18,869 patients
- Median age: 62 years (range: 0-95)
- Sex distribution: 52% male, 48% female

### Diagnostic Label Distribution:
- **Class imbalance** detected:
  - NORM (Normal): ~9,500 records (largest)
  - HYP (Hypertrophy): ~2,600 records (smallest)
- Multi-label records present (one ECG can have multiple diagnoses)

### Signal Quality:
- Some records have quality issues (noise, baseline drift, electrode problems)
- Need quality filtering in preprocessing

### Next Steps:
1. Data preprocessing and cleaning
2. Signal filtering and noise removal
3. Time series decomposition
4. Feature extraction